In [ ]:
import os
import json
import shutil
import subprocess
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from dotenv import load_dotenv
from pathlib import Path
from functools import wraps
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()

# LLMClient

In [ ]:
class LLMClient:
    def __init__(self):
        self._client = ChatOpenAI(
            base_url=os.getenv("OPENAI_API_HOST"),
            api_key=os.getenv("OPENAI_API_KEY"),
            model="openai/gpt-oss-20b",
            timeout=None,
        )

    def generate(self, prompt: str) -> str:
        return self._client.invoke(prompt).content

# AgentContext — общая память

In [ ]:
@dataclass
class AgentContext:
    requirements: str #
    plan: list[dict] = field(default_factory=list) #план от Менеджера
    domain: str | None = None       # результат DDD только в памяти
    tests: str | None = None        # результат TDD только в памяти
    review: str | None = None       # результат Reviewer только в памяти
    files: dict[str, str] = field(default_factory=dict)  # html/js/csv → пишутся на диск

# Декоратор agent_step

In [ ]:
# ===== Декоратор: логирование + is_done =====

def agent_step(name: str):
    def decorator(run_fn):
        @wraps(run_fn)
        def wrapper(self, ctx: AgentContext) -> AgentContext:
            print(f"\n▶ [{name}] started")
            ctx = run_fn(self, ctx)
            for step in ctx.plan:
                if step.get("executor") == name and not step["is_done"]:
                    step["is_done"] = True
                    break
            print(f"✅ [{name}] done")
            return ctx
        return wrapper
    return decorator

# Делает три вещи: печатает старт, запускает агента, ставит is_done: True в плане.

# Tools — три инструмента

In [ ]:
@tool
def run_docker_compose(compose_path: str = "repo/docker-compose.yml") -> str:
    """Run docker compose up --build -d for given compose file path."""
    command = f"docker compose -f {compose_path} up --build -d".split()
    result = subprocess.run(command, capture_output=True, text=True)
    output = result.stdout + result.stderr
    if result.returncode == 0:
        return f"✅ Docker launched.\n{output[:500]}"
    return f"❌ Docker failed (code {result.returncode}):\n{output[:500]}"


@tool
def check_docker_status() -> str:
    """Check running docker containers."""
    result = subprocess.run(["docker", "ps"], capture_output=True, text=True)
    return result.stdout


@tool
def write_output_files(files_json: str) -> str:
    """Write only .html, .js, .csv files to OUTPUT_DIR. Everything else is skipped."""
# Получает JSON со всеми файлами
# Фильтрует — пишет ТОЛЬКО .html .js .csv
# Остальное пропускает с ⏭
    output_dir = Path(os.getenv("OUTPUT_DIR", "demo"))
    shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    files = json.loads(files_json)
    written = []
    for filename, content in files.items():
        ext = Path(filename).suffix.lower()
        if ext in (".html", ".js", ".csv"):
            (output_dir / filename).write_text(content, encoding="utf-8")
            written.append(filename)
        else:
            print(f"  ⏭ Skipped: {filename}")
    return f"Written to {output_dir}: {written}"

### - run_docker_compose — запускает контейнер
### -check_docker_status — проверяет запущенные контейнеры
### - write_output_files — пишет файлы на диск
### - Папка берётся из OUTPUT_DIR в .env, по умолчанию demo/.

# BaseAgent - шаблон

In [ ]:
class BaseAgent(ABC):
    def __init__(self, llm: LLMClient):
        self.llm = llm

    @abstractmethod
    def run(self, ctx: AgentContext) -> AgentContext:
        ...

### `ABC` + `@abstractmethod` означает — нельзя создать агента без метода `run()`. Python выдаст ошибку. Все 6 агентов наследуются от него.

# Шесть агентов

---

### ManagerAgent — строит план
- Промт требует JSON-массив, каждый executor ровно **один раз**
- Если LLM вернул не JSON → fallback план из 5 шагов
- Результат: `ctx.plan`

---

In [ ]:
# ===== 1. Manager =====

class ManagerAgent(BaseAgent):
    TOOLS = [run_docker_compose, check_docker_status, write_output_files]

    @agent_step("Manager")
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"""
Ты менеджер мультиагентной системы для создания веб-приложений.
Составь план разработки в виде JSON-массива.

Каждый шаг: {{"action": "описание задачи", "is_done": false, "executor": "AgentName"}}

Допустимые executor — используй каждый РОВНО ОДИН РАЗ:
- DDD       — анализ доменной модели
- TDD       — написание тестов
- Developer — генерация HTML/JS/CSV
- Reviewer  — ревью кода
- Executor  — запуск docker

Требования к приложению: {ctx.requirements}

Верни ТОЛЬКО JSON-массив. Без markdown, без пояснений.
"""
        raw = self.llm.generate(prompt)
        raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            ctx.plan = json.loads(raw)
        except json.JSONDecodeError:
            ctx.plan = [
                {"action": "Анализ доменной модели", "is_done": False, "executor": "DDD"},
                {"action": "Написание тестов", "is_done": False, "executor": "TDD"},
                {"action": "Генерация кода", "is_done": False, "executor": "Developer"},
                {"action": "Ревью кода", "is_done": False, "executor": "Reviewer"},
                {"action": "Запуск", "is_done": False, "executor": "Executor"},
            ]
        print(f"  Plan:\n{json.dumps(ctx.plan, ensure_ascii=False, indent=2)}")
        return ctx

---
### DDDAgent — доменная модель
- Читает: `ctx.requirements`
- Просит: сущности + связи + бизнес-правила
- Результат: `ctx.domain` *(только память)*

---

In [ ]:
# ===== 2. DDD Agent =====

class DDDAgent(BaseAgent):

    @agent_step("DDD")
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"""
Ты аналитик доменных моделей.
Опиши доменную модель для веб-приложения по требованиям ниже.

Требования: {ctx.requirements}

Включи:
- Основные сущности и их атрибуты
- Связи между сущностями
- Ключевые бизнес-правила

Отвечай структурированно и кратко — это будет использовано для генерации кода.
"""
        ctx.domain = self.llm.generate(prompt)
        print(f"  Domain model → ctx.domain (in memory)")
        return ctx

---
###  TDDAgent — тесты
- Читает: `ctx.requirements` + `ctx.domain`
- Просит: pytest без внешних библиотек
- Результат: `ctx.tests` *(только память)*

---

In [ ]:
# ===== 3. TDD Agent =====

class TDDAgent(BaseAgent):

    @agent_step("TDD")
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"""
Ты инженер по тестированию.
Напиши pytest-тесты для веб-приложения.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

Правила:
- Покрой основную бизнес-логику
- Используй только стандартные библиотеки Python и pytest
- Верни только Python-код без markdown и пояснений
"""
        ctx.tests = self.llm.generate(prompt)
        print(f"  Tests → ctx.tests (in memory)")
        return ctx

---
###  DeveloperAgent — генерирует 3 файла 

| Промт | Правила |
|-------|---------|
| `html_prompt` | валидный HTML, стили в `<style>`, Chart.js если нужны графики, id/class для всех элементов, `script.js` в конце body |
| `js_prompt` | `DOMContentLoaded`, демо-данные в коде (8-10 записей), `createElement` + `textContent`, вся интерактивность |
| `csv_prompt` | только CSV, 20+ строк, даты `YYYY-MM-DD` |

- Результат: `ctx.files` + файлы на диск через `write_output_files`

---

In [ ]:
# ===== 4. Developer Agent =====

class DeveloperAgent(BaseAgent):

    @agent_step("Developer")
    def run(self, ctx: AgentContext) -> AgentContext:

        # --- HTML ---
        html_prompt = f"""
Ты frontend-разработчик. Создай одностраничное веб-приложение.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

СТРОГИЕ ПРАВИЛА:
1. Верни ТОЛЬКО валидный HTML от <!DOCTYPE html> до </html>
2. Без markdown, без пояснений, без блоков ```
3. Все стили — внутри <style> в <head>. Дизайн: современный, чистый, адаптивный
4. Все интерактивные элементы должны иметь id или class — для управления из script.js
5. Если приложение предполагает графики — добавь:
   <script src="https://cdn.jsdelivr.net/npm/chart.js@4.3.3/dist/chart.umd.min.js"></script>
   и <canvas id="mainChart"></canvas> в нужном месте
6. В конце <body> подключи: <script src="script.js"></script>
7. Структура и содержимое страницы должны точно соответствовать требованиям
8. Все теги закрыты. HTML валиден.
"""

        # --- JS ---
        js_prompt = f"""
Ты JavaScript-разработчик. Создай script.js для веб-приложения.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

СТРОГИЕ ПРАВИЛА:
1. Верни ТОЛЬКО чистый JS-код без markdown и блоков ```
2. Весь код оберни в: document.addEventListener('DOMContentLoaded', () => {{ /* весь код */ }})
3. Создай демо-данные прямо в коде — массивы объектов, соответствующие доменной модели
   Данные должны быть реалистичными и соответствовать типу приложения
   Минимум 8-10 записей
4. Для вставки данных в DOM используй createElement + textContent (не innerHTML с данными)
5. Реализуй всю интерактивность из требований: фильтры, сортировка, поиск, кнопки и т.д.
6. Если в HTML есть canvas#mainChart — создай Chart.js график подходящего типа
   (Chart уже подключён глобально, не импортируй его)
7. Не используй fetch, axios или внешние запросы — только локальные данные
8. Код должен работать сразу после открытия index.html в браузере
"""

        # --- CSV ---
        csv_prompt = f"""
Создай CSV-файл с демонстрационными данными для веб-приложения.

Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

ПРАВИЛА:
1. Верни ТОЛЬКО CSV-текст без markdown и блоков ```
2. Первая строка — заголовки колонок, соответствующие доменной модели
3. Минимум 20 строк реалистичных данных
4. Форматы: даты YYYY-MM-DD, числа без пробелов, строки без лишних кавычек
"""

        print(f"  Generating HTML...")
        index_html = self.llm.generate(html_prompt)

        print(f"  Generating JS...")
        script_js = self.llm.generate(js_prompt)

        print(f"  Generating CSV...")
        data_csv = self.llm.generate(csv_prompt)

        ctx.files = {
            "index.html": index_html,
            "script.js":  script_js,
            "data.csv":   data_csv,
        }

        result = write_output_files.invoke({"files_json": json.dumps(ctx.files, ensure_ascii=False)})
        print(f"  {result}")
        return ctx


---
### ReviewerAgent — проверяет код
- Читает: первые 1200 символов `html` и `js` из `ctx.files`
- Находит: до 10 проблем с решениями
- Результат: `ctx.review` *(только память)*

---


In [ ]:
# ===== 5. Reviewer Agent =====

class ReviewerAgent(BaseAgent):

    @agent_step("Reviewer")
    def run(self, ctx: AgentContext) -> AgentContext:
        html_snippet = ctx.files.get("index.html", "")[:1200]
        js_snippet   = ctx.files.get("script.js", "")[:1200]
        prompt = f"""
Ты senior-разработчик. Проведи code review сгенерированного веб-приложения.

Требования к приложению: {ctx.requirements}

HTML (фрагмент):
{html_snippet}

JS (фрагмент):
{js_snippet}

Найди конкретные проблемы и предложи улучшения (до 10 пунктов).
Формат каждого пункта: номер. Проблема → Решение
"""
        ctx.review = self.llm.generate(prompt)
        print(f"  Review → ctx.review (in memory)")
        return ctx

---
###  ExecutorAgent — запускает Docker
- Проверяет: существует ли `repo/docker-compose.yml`
- Если **нет** → пропускает с предупреждением
- Если **да** → вызывает `run_docker_compose`
---

In [ ]:
# ===== 6. Executor Agent =====

class ExecutorAgent(BaseAgent):

    @agent_step("Executor")
    def run(self, ctx: AgentContext) -> AgentContext:
        compose_path = "repo/docker-compose.yml"
        if not Path(compose_path).exists():
            print(f"  ⚠ {compose_path} not found, skipping docker launch")
            return ctx
        result = run_docker_compose.invoke({"compose_path": compose_path})
        print(f"  {result[:300]}")
        return ctx

### EXECUTOR_MAP

In [ ]:
# ===== Executor Map =====

EXECUTOR_MAP = {
    "DDD":       DDDAgent,
    "TDD":       TDDAgent,
    "Developer": DeveloperAgent,
    "Reviewer":  ReviewerAgent,
    "Executor":  ExecutorAgent,
}


### Словарь связывает строку из плана менеджера с классом агента. 

# Pipeline 

In [ ]:
def run_pipeline(prompt: str, llm: LLMClient | None = None) -> AgentContext:
    if llm is None:
        llm = LLMClient()

    ctx = AgentContext(requirements=prompt)

    # Шаг 0: Менеджер строит план
    ctx = ManagerAgent(llm).run(ctx)

    # Шаги 1-N: каждый executor запускается только один раз
    seen = set() # защита от дублей
    for step in ctx.plan:
        executor_name = step.get("executor")
        if executor_name in seen:
            print(f"  ⏭ {executor_name} already ran, skipping")
            continue
        seen.add(executor_name)
        AgentClass = EXECUTOR_MAP.get(executor_name)
        if AgentClass:
            ctx = AgentClass(llm).run(ctx)
        else:
            print(f"  ⚠ Unknown executor: {executor_name}, skipping")

    print("\n Pipeline complete!")
    print(f"   Files on disk : {list(ctx.files.keys())}")
    print(f"   Domain in mem : {bool(ctx.domain)}")
    print(f"   Tests  in mem : {bool(ctx.tests)}")
    print(f"   Review in mem : {bool(ctx.review)}")
    return ctx

In [7]:
prompt = "Создай интерактивный дашборд с таблицей данных и графиком продаж."
run_pipeline(prompt);


▶ [Manager] started
  Plan:
[
  {
    "action": "DDD: Проанализировать доменную модель, определить сущности SalesRecord, DashboardConfig, DataSource, а также связи и бизнес‑правила для интерактивного дашборда",
    "is_done": false,
    "executor": "DDD"
  },
  {
    "action": "TDD: Написать модульные тесты для всех бизнес‑логик (форматирование данных, фильтрация, агрегация) и для UI‑компонентов (таблица, график), используя Jest и React Testing Library",
    "is_done": false,
    "executor": "TDD"
  },
  {
    "action": "Developer: Сгенерировать HTML/JS/CSV – построить шаблон с таблицей данных, внедрить график продаж с Chart.js, добавить реактивные состояния и стили Bootstrap",
    "is_done": false,
    "executor": "Developer"
  },
  {
    "action": "Reviewer: Осуществить ревью кода – проверить соответствие бизнес‑правилам, структуре компонентов, читаемость, покрытие тестами",
    "is_done": false,
    "executor": "Reviewer"
  },
  {
    "action": "Executor: Создать Dockerfile, собрат